# 03 - Data Analysis

## Overview

Deep dive into data: feature importance, correlation analysis, class distribution, outlier detection.

**Run after**: `02_preprocessing.ipynb`

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '/home/willtanoe/Documents/fl-xgb-thesis')
from src.config import SEED

# Paths
PREPROCESSED_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/preprocessed"
FIGURES_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/figures"
LOGS_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/logs"

os.makedirs(FIGURES_DIR, exist_ok=True)

print("Data Analysis notebook initialized.")

## 1. Load Preprocessed Data

In [ ]:
print("Loading preprocessed data...")
train_df = pd.read_parquet(os.path.join(PREPROCESSED_DIR, "train.parquet"))

# Load metadata
with open(os.path.join(PREPROCESSED_DIR, "metadata.json"), 'r') as f:
    metadata = json.load(f)

feature_cols = metadata['feature_cols']
class_names = metadata['classes']
num_classes = metadata['num_classes']

print(f"Train shape: {train_df.shape}")
print(f"Features: {len(feature_cols)}")
print(f"Classes: {num_classes}")

## 2. Feature Importance (XGBoost)

In [ ]:
import xgboost as xgb
from sklearn.utils import resample

print("Training XGBoost for feature importance...")

# Subsample to avoid OOM on 16GB RAM
sample_size = min(200_000, len(train_df))
X_samp, y_samp = resample(
    train_df[feature_cols].values,
    train_df['label'].values,
    n_samples=sample_size,
    random_state=42
)
print(f"Using {sample_size:,} samples for feature importance")

model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=num_classes,
    n_estimators=100,
    max_depth=6,
    tree_method='hist',
    device='cpu',
    random_state=SEED,
    verbosity=0
)
model.fit(X_samp, y_samp)

# Get feature importance
importance = model.feature_importances_
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': importance
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features:")
print(importance_df.head(20).to_string(index=False))

In [ ]:
# Plot feature importance
fig, ax = plt.subplots(figsize=(12, 10))

top_n = 30
top_features = importance_df.head(top_n)

ax.barh(range(top_n), top_features['importance'].values, color='steelblue')
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_features['feature'].values)
ax.invert_yaxis()
ax.set_xlabel('Importance Score')
ax.set_title(f'Top {top_n} Feature Importance (XGBoost)')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '03_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Figure saved: results/figures/03_feature_importance.png")

## 3. Correlation Analysis

In [ ]:
# Correlation matrix for top features
top_20_features = importance_df.head(20)['feature'].tolist()

corr_matrix = train_df[top_20_features].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 8})
ax.set_title('Correlation Matrix (Top 20 Features)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '03_correlation_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

# Find highly correlated pairs
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.9:
            high_corr.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

if high_corr:
    print("\nHighly correlated pairs (>0.9):")
    for f1, f2, corr in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True):
        print(f"  {f1} <-> {f2}: {corr:.3f}")
else:
    print("\nNo highly correlated pairs (>0.9) found.")

## 4. Class Distribution Deep Dive

In [ ]:
# Detailed class distribution
label_counts = train_df['label'].value_counts()
label_pct = (label_counts / len(train_df) * 100).round(4)

class_dist = pd.DataFrame({
    'class_id': label_counts.index.tolist(),
    'class_name': [class_names[c] for c in label_counts.index],
    'count': label_counts.values,
    'percentage': label_pct.values
})

print("Class Distribution:")
print(class_dist.to_string(index=False))

In [ ]:
# Class distribution categories
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Major classes (>1%)
major = class_dist[class_dist['percentage'] > 1]
ax1 = axes[0]
ax1.bar(range(len(major)), major['percentage'].values, color='steelblue')
ax1.set_xticks(range(len(major)))
ax1.set_xticklabels([f"{i}\n{name[:15]}" for i, name in zip(major['class_id'], major['class_name'])], 
                     rotation=45, ha='right', fontsize=8)
ax1.set_ylabel('Percentage (%)')
ax1.set_title('Major Classes (>1%)')
ax1.grid(axis='y', alpha=0.3)

# Minor classes (<1%)
minor = class_dist[class_dist['percentage'] <= 1]
ax2 = axes[1]
ax2.bar(range(len(minor)), minor['percentage'].values, color='coral')
ax2.set_xticks(range(len(minor)))
ax2.set_xticklabels([f"{i}\n{name[:10]}" for i, name in zip(minor['class_id'], minor['class_name'])], 
                     rotation=45, ha='right', fontsize=7)
ax2.set_ylabel('Percentage (%)')
ax2.set_title('Minor Classes (<=1%)')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '03_class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure saved: results/figures/03_class_distribution.png")

## 5. Feature Statistics by Class

In [ ]:
# Mean feature values per class (top 5 features)
top_features = importance_df.head(5)['feature'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    ax = axes[i]
    class_means = train_df.groupby('label')[feat].mean().sort_values(ascending=False)
    ax.bar(range(len(class_means)), class_means.values, alpha=0.7)
    ax.set_title(f'{feat} (Mean by Class)')
    ax.set_xticks(range(len(class_means)))
    ax.set_xticklabels(range(len(class_means)), fontsize=6)
    ax.grid(axis='y', alpha=0.3)

axes[-1].axis('off')  # Hide last empty subplot

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '03_feature_by_class.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Figure saved: results/figures/03_feature_by_class.png")

## 6. Outlier Detection

In [ ]:
from scipy import stats

# Z-score analysis for top features
top_features = importance_df.head(10)['feature'].tolist()

outlier_summary = []
for feat in top_features:
    z_scores = np.abs(stats.zscore(train_df[feat]))
    outliers = (z_scores > 3).sum()
    outlier_pct = outliers / len(train_df) * 100
    outlier_summary.append({
        'feature': feat,
        'outliers': outliers,
        'outlier_pct': outlier_pct
    })

outlier_df = pd.DataFrame(outlier_summary)
print("Outlier Analysis (Z-score > 3) for Top Features:")
print(outlier_df.to_string(index=False))

## 7. Save Analysis Summary

In [ ]:
# Save analysis results
analysis_results = {
    'top_features': importance_df.head(30).to_dict('records'),
    'class_distribution': class_dist.to_dict('records'),
    'num_features': len(feature_cols),
    'num_classes': num_classes,
    'outlier_summary': outlier_df.to_dict('records')
}

with open(os.path.join(LOGS_DIR, '03_data_analysis.json'), 'w') as f:
    json.dump(analysis_results, f, indent=2)

print("Analysis saved: results/logs/03_data_analysis.json")

## Summary

- Feature importance computed via XGBoost
- Correlation analysis shows feature relationships
- Class distribution: major vs minor classes identified
- Feature statistics by class
- Outlier detection via Z-score

**Next**: Proceed to `04_fl_setup.ipynb`